In [5]:
# 4장의 FK를 위한 사전 정의 요소들

# 1. UR5 6DOF 로봇 파라미터 단위 (mm)
L1 = 425
L2 = 392
W1 = 109
W2 = 82
H1 = 89
H2 = 95

# 2. 모든 조인트가 0일 때의 EE 좌표계 변환 행렬 M
# 책의 222p 6DOF UR5 로봇 기반 M 행렬로 작성

import numpy as np

R = np.array([[-1, 0, 0],
              [ 0, 0, 1],
              [ 0, 1, 0]])

p = np.array([[L1 + L2],
              [W1 + W2],
              [H1 - H2]])

M = np.block([[R,              p],
             [np.zeros((1, 3)), np.array([[1]])]])

# 3. thetalist -> theta_2: -90, theta_5: 90
thetalist = [0, -np.pi/2, 0, 0, np.pi/2, 0]

# 4. Blist

# 공간꼴 PoE
w1_space = np.array([0, 0, 1])
w2_space = np.array([0, 1, 0])
w3_space = np.array([0, 1, 0])
w4_space = np.array([0, 1, 0])
w5_space = np.array([0, 0, -1])
w6_space = np.array([0, 1, 0])

# 물체꼴 PoE
w1_body = np.array([0, 1, 0])
w2_body = np.array([0, 0, 1])
w3_body = np.array([0, 0, 1])
w4_body = np.array([0, 0, 1])
w5_body = np.array([0, -1, 0])
w6_body = np.array([0, 0, 1])

# 스크류 축 선상의 한점 q
# q1과 q2는 동일한 원점을 공유 가능
q1_space = np.array([0, 0, 0])
q2_space = np.array([0, 0, 0])
q3_space = np.array([L1, W1, -H1])
q4_space = np.array([L2, 0, 0])
q5_space = np.array([0, W1 - W2, 0])
q6_space = np.array([0, 0, -H2])

q1_body = np.array([0, 0, -W1])
q2_body = np.array([L1, 0, 0])
q3_body = np.array([L2, 0, 0])
q4_body = np.array([0, H2, 0])
q5_body = np.array([0, 0, -W2])
q6_body = np.array([0, 0, 0])

# [w] skew-symmetric 형태를 위한 유틸리티
def skew(w):
    return np.array([[    0, -w[2],  w[1]],
                     [ w[2],     0, -w[0]],
                     [-w[1],  w[0],     0]])

# v = -w x q (외적)
v1_space = -np.cross(w1_space, q1_space)
v2_space = -np.cross(w2_space, q2_space)
v3_space = -np.cross(w3_space, q3_space)
v4_space = -np.cross(w4_space, q4_space)
v5_space = -np.cross(w5_space, q5_space)
v6_space = -np.cross(w6_space, q6_space)

v1_body = -np.cross(w1_body, q1_body)
v2_body = -np.cross(w2_body, q2_body)
v3_body = -np.cross(w3_body, q3_body)
v4_body = -np.cross(w4_body, q4_body)
v5_body = -np.cross(w5_body, q5_body)
v6_body = -np.cross(w6_body, q6_body)

# 스크류 [S_i] se(3) 4x4 행렬: [[w], v; 0, 0]
def se3(w, v):
    return np.block([[skew(w), v.reshape(3, 1)],
                     [np.zeros((1, 4))]])

S1_space = se3(w1_space, v1_space)
S2_space = se3(w2_space, v2_space)
S3_space = se3(w3_space, v3_space)
S4_space = se3(w4_space, v4_space)
S5_space = se3(w5_space, v5_space)
S6_space = se3(w6_space, v6_space)

S1_body = se3(w1_body, v1_body)
S2_body = se3(w2_body, v2_body)
S3_body = se3(w3_body, v3_body)
S4_body = se3(w4_body, v4_body)
S5_body = se3(w5_body, v5_body)
S6_body = se3(w6_body, v6_body)

Slist_space = [S1_space, S2_space, S3_space, S4_space, S5_space, S6_space]
Blist_body = [S1_body, S2_body, S3_body, S4_body, S5_body, S6_body]


In [6]:
# 3.12 동차 변환행렬 T의 6x6 수반 표현 [Ad_T]를 계산
# 입력: T  4x4
# 출력 AdT 6x6

def Adjoint(T):
    R, p = Trans2Rp(T)
    # Vec2so3 함수 활용
    p_mat = Vec2so3(p)

    AdT = np.block([[R,                  np.zeros((3, 3))],
                   [np.dot(p_mat, R),   R               ]])

    return AdT

In [7]:
# forward kinematics의 스크류 운동 기반 해석 (D-H 기반 해석 아님)
# 4.1 영 위치에서의 EE 좌표계 M이 주어지고 EE 좌표계 기준 PoE 해석

# 입력: 관절 스크류 Blist (nx1), 관절 값 thetalist (nx1)
# 출력: EE 좌표계의 pose

from scipy.linalg import expm

def body_frame_fk(Blist_body, thetalist):

    T_body = np.copy(M)
    for B, theta in zip(Blist_body, thetalist):
        T_body = T_body @ expm(B * theta)

    return T_body


In [8]:
# 4.2 영 위치에서의 EE 좌표계 M이 주어지고 고정 좌표계 기준 PoE 해석

# 입력: 관절 스크류 Slist (nx1), 관절 값 thetalist (nx1)
# 출력: EE 좌표계의 pose

def fixed_frame_fk(Slist_space, thetalist):

    T_space = np.eye(4)
    for S, theta in zip(Slist_space, thetalist):
        T_space = T_space @ expm(S * theta)
    T_space = T_space @ M

    return T_space

In [9]:
T_body = body_frame_fk(Blist_body, thetalist)
T_space = fixed_frame_fk(Slist_space, thetalist)
print(T_body)
print(T_space)

[[-1.36002321e-15 -1.00000000e+00 -6.80011603e-16  3.92000000e+02]
 [ 1.00000000e+00 -1.65666092e-16 -3.31332184e-16  1.09000000e+02]
 [-1.67574288e-15 -2.16840434e-16  1.00000000e+00  5.01000000e+02]
 [-3.46944695e-18 -8.67361738e-19 -1.73472348e-18  1.00000000e+00]]
[[ 0.00000000e+00 -1.00000000e+00  0.00000000e+00  6.00000000e+00]
 [ 1.00000000e+00  0.00000000e+00  0.00000000e+00 -7.90000000e+02]
 [-2.22044605e-16  0.00000000e+00  1.00000000e+00  1.64000000e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
